# Twin / Course Notebook 1 — Supervised Fine-Tuning with QLoRA

Fine-tune a small open model on YOUR instruction dataset using QLoRA, on a **free Colab T4 GPU**.

**Before you run:** Runtime -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

What you'll do: load a small base model in 4-bit, attach LoRA adapters, train on a tiny instruction set,
and save the adapter. Every cell has a markdown note explaining what it does and what to watch.


## 1. Install the libraries
Unsloth makes QLoRA fast and memory-light on a T4. This takes a minute or two.


In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

## 2. Load a small base model in 4-bit
We load a ~1-3B model quantized to 4-bit (the QLoRA trick: the frozen base needs no high precision).
Watch the printed memory footprint — this is what makes a free GPU enough.


In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",  # small enough for a free T4
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,   # QLoRA: load the frozen base in 4-bit
)

## 3. Attach LoRA adapters
We freeze the whole base model and add small trainable adapters (<1% of params).
`r` is the adapter rank (size): bigger = more capacity, more memory. 8-32 is typical.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

## 4. Load YOUR dataset
Upload a `train.jsonl` where each line is a chat example:
`{"messages": [{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}`
(This is the format you built in the course. For the Twin, these are your-voice Q->A pairs.)


In [ ]:
from datasets import load_dataset

# Upload train.jsonl via the Colab file panel on the left, then:
dataset = load_dataset("json", data_files="train.jsonl", split="train")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat)
print(dataset[0]["text"][:500])
print("examples:", len(dataset))

## 5. Train
Watch the **loss** in the output: it should trend **down**. If it's flat, your learning rate or data is off;
if it crashes to ~0 immediately, you're likely memorizing a tiny set (raise data, lower epochs).


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,        # 1-3 is plenty; more memorizes
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)
trainer.train()

## 6. Try it, then save the adapter
Generate from the tuned model, then save the tiny adapter (megabytes) for reuse / serving (Level 5).


In [ ]:
FastLanguageModel.for_inference(model)
messages = [{"role":"user","content":"Write a two-sentence intro about me."}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt",
                                       add_generation_prompt=True).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))

model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")
print("Saved adapter to ./lora_adapter — download it from the file panel.")

---
**You just ran a real QLoRA fine-tune on a free GPU.** Next: Notebook 2 (DPO) refines judgment,
Notebook 3 evaluates base vs tuned. Back in the course, paste your eval JSON to verify this lab.
